In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
vp_data_db = "../example_data/vp_data_actors.db"

trans_actors = "patterns_transaction_actors_alati"

### root+verbimuster -> count mustrid tabel 'alati' ja 'mitte kunagi' verbide jaoks

In [3]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()

# alati root+verb+kaane+head_cnt

In [4]:
query = """
SELECT root_word, verb_word||'_'||verb_compound||'_'||phrase_case as verb_pat, count(distinct head_id) as head_cnt 
FROM {tbl}
group by root_word, verb_pat
order by head_cnt desc
""".format(tbl=trans_actors)

sb = pd.read_sql_query(query, con)
sb

,root_word,verb_pat,head_cnt
0,mina,meeldima__all,2
1,mina,tekkima__ad,2
2,mina,helistama__all,1
3,mina,imponeerima__all,1
4,mina,minema_peale_all,1
5,mina,nõudma__abl,1
6,rahvas,minema_peale_all,1
7,sina,tulema__ad,1
8,tema,tulema__ad,1
9,tema,tundma_kaasa_all,1


# These are results from original table with more data

### mis on top 100 mustrit ja kui palju roote need ära katavad

In [59]:
verb_pats100 = list(sb[:100]["verb_pat"])
verb_pats100roots = list(set(list(sb[:100]["root_word"])))

In [60]:
len(verb_pats100roots)

37

In [64]:
len(list(set(list(sb["root_word"]))))

72281

# alati root + verb_pat count

In [20]:
query = """

select root_word, count(verb_pat) as verb_count from (
SELECT root_word, verb_word||'_'||phrase_case as verb_pat--, count(distinct head_id) as head_cnt 
FROM {tbl}
group by root_word, verb_pat
--order by head_cnt desc
) as tbl1
group by root_word
order by verb_count desc
""".format(tbl=trans_actors)
source = pd.read_sql_query(query, con)

In [21]:
source

,root_word,verb_count
0,tema,402
1,mina,398
2,sina,301
3,kes,282
4,ise,269
...,...,...
72276,0-millennium,1
72277,"0,5",1
72278,-6%,1
72279,-2%,1


## alati top 100 root

In [6]:
list(source.iloc[:100]["root_word"])

['tema',
 'mina',
 'sina',
 'kes',
 'ise',
 'see',
 'inimene',
 'mees',
 'teine',
 'naine',
 'laps',
 'keegi',
 'kõik',
 'rahvas',
 'mis',
 'riik',
 'sõber',
 'maa',
 'ema',
 'Venemaa',
 'ajakirjanik',
 'eestlane',
 'juht',
 'poiss',
 'Eesti',
 'tee',
 'klient',
 'üksteise',
 'isa',
 'firma',
 'valitsus',
 'oma',
 'president',
 'üks',
 'linn',
 'tüdruk',
 'töötaja',
 'töö',
 'tänav',
 'politsei',
 'külaline',
 'noor',
 'auto',
 'pool',
 'vanem',
 'omanik',
 'liige',
 'politseinik',
 'õpilane',
 'tütar',
 'publik',
 'soomlane',
 'õpetaja',
 'vend',
 'kolleeg',
 'vaataja',
 'noormees',
 'koht',
 'elanik',
 'lugeja',
 'kodanik',
 'turg',
 'abikaasa',
 'poeg',
 'pere',
 'isik',
 'arst',
 'teineteise',
 'lava',
 'pank',
 'iseenese',
 'Saksamaa',
 'meeskond',
 'ettevõte',
 'ametnik',
 'ameeriklane',
 'maailm',
 'küsimus',
 'võim',
 'saar',
 'ala',
 'venelane',
 'jõud',
 'vastane',
 'naaber',
 'ostja',
 'näitleja',
 'poja',
 'minister',
 'kaaslane',
 'autor',
 'tuttav',
 'peaminister',
 'mäng

## alati top 100 mustrite arvu alusel elusad

### precision = 75-87% on elusad
kui lubada teine,maa,firma, töö, politsei,maailm, pank, võim, ettevõte, jõud,turg,linn,

### elusad ka riigid: 
tema, mina, sina, kes, ise, mees, inimene, naine, laps, keegi, rahvas, sõber, riik, ajakirjanik, juht, ema, eestlane, Venemaa, poiss, klient, Eesti, üksteise, isa, president, valitsus, oma, külaline, töötaja, tüdruk, vanem, noor, publik, politseinik, liigem omanik, õpilane, vend, tütar, soomlane, kolleeg, õpetaja, vaataja, lugeja, noormees, elanik, teineteise, kodanik, abikaasa, poeg, pere, arst, isik, iseenese, ameeriklane, meeskond, ametnik, Saksamaa, venelane, ostja, kaaslane, minister, vastane, naaber, näitleja, poja, peremees, peaminister, autor, mängija, poliitik, kohtunik, tuttav, ohver, liider, tudeng

### mis ei ole elus ega ka agent või on kahtlane: 
see,kõik,mis,tee,üks,tänav,auto,pool,koht,küsimus,lava,saar,ala, 

teine,maa,firma, töö, politsei,maailm, pank, võim, ettevõte, jõud,turg,linn,

# alati verbi esinemistele vastav rootide arv

### st kui palju on roote (root_word) kui verb_count on 1, 2, 3, 4 jne

In [7]:
df1 = pd.DataFrame(source.groupby('verb_count')['root_word'].nunique())
df1

,root_word
verb_count,
1,44945
2,9281
3,4452
4,2598
5,1779
...,...
269,1
282,1
301,1


In [8]:
# kui palju roote esineb ainult 1 mustris: 44945
source[source["verb_count"]==1]

,root_word,verb_count
27336,žüriihääletus,1
27337,žurnalistikatudeng,1
27338,žukov,1
27339,žtaalia,1
27340,žnternationa,1
...,...,...
72276,0-millennium,1
72277,"0,5",1
72278,-6%,1
72279,-2%,1


In [9]:
# kui palju roote esineb vähemalt 10 mustris: 5,517
source[source["verb_count"]>=10]

,root_word,verb_count
0,tema,402
1,mina,398
2,sina,301
3,kes,282
4,ise,269
...,...,...
5512,Angela,10
5513,Andre,10
5514,Aleksejeva,10
5515,AGA,10


In [10]:
# kui palju roote esineb vähemalt 100 mustris: 111
source[source["verb_count"]>=100]

,root_word,verb_count
0,tema,402
1,mina,398
2,sina,301
3,kes,282
4,ise,269
...,...,...
106,iga,101
107,avalikkus,101
108,tudeng,100
109,loom,100


In [65]:
# verb_pats100roots = top100 sagedasema verb+kääne rootid
roots_atleast100 = list(set(list(source[source["verb_count"]>=100]["root_word"])))

In [67]:
yhisosa = []

for w in verb_pats100roots:
    if w in roots_atleast100:
        yhisosa.append(w)

In [68]:
len(yhisosa)

14

In [5]:
con.close()